# Log search and detection as code

Run one hunt in SPL, KQL and SQL over real-format logs, parse a raw log into fields, then turn the hunt into Sigma rules you test for hits and false positives, tune, and convert with pySigma.

The files for this seminar are in `~/case` (Linux) and `case/` (Python): `access.log`, `auth.log`, `eventlogs/Security.jsonl`, `shop2_access.log`.

## 1. One question, three languages

A hunt starts with a question, not a tool. Ours: **did anyone guess an SSH password on web01, get in, and do something with it?** You'll answer it three times: in SPL, in KQL and in SQL. Real teams use different SIEMs, and moving a hunt between them is a daily job.

The SIEM search here is a **teaching subset** of each language over the case logs, so not every command works. When you want the real thing, the free [KC7](https://kc7cyber.com) game and the free Azure Data Explorer help cluster run full KQL.

First look at the raw log with `grep`, so you know what the search engine is reading.

- [ ] How many lines does auth.log have?
- [ ] Which line types do you see: Failed, Accepted, sudo?
- [ ] Which address keeps failing?

> Hint: `grep -c` counts matching lines. `awk '{print $(NF-3)}'` prints the fourth word from the end, which is the source address on every Failed line.

?> How many lines in auth.log contain "Failed password"? :: sha256:7902699be42c8a8e46fbbb4501726517e86b22c56a189f7625a6da49081b2451

In [ ]:
%%bash
cd ~/case
wc -l auth.log access.log
grep -c 'Failed password' auth.log
grep 'Failed password' auth.log | awk '{print $(NF-3)}' | sort | uniq -c | sort -rn

## 2. SPL: count, then pivot

SPL reads left to right: a search, then a pipe `|`, then a command that works on the results. `stats count by src_ip` groups the failures by address, and `sort - count` puts the biggest first.

Then **pivot**: take the address you found and search for it again, this time for a success.

- [ ] Run the first search and read the table
- [ ] Change it to find the Accepted login from the same address
- [ ] Find what that user did with sudo

> Hint: Field names come from the parsed log: `action`, `src_ip`, `user`, `auth_method`. Click any field name in the results to add it to the search.

```tool siem
{"q": "sourcetype=syslog action=failure | stats count by src_ip | sort - count", "n": 2, "files": ["auth.log"], "blank": ["q"]}
```

```tool siem
{"q": "Accepted src_ip=203.0.113.7 | table _time user auth_method", "n": 1, "files": ["auth.log"], "blank": ["q"]}
```

```tool siem
{"q": "process=sudo | table _time user run_as command", "n": 1, "files": ["auth.log"], "blank": ["q"]}
```

?> Which address failed most often? :: sha256:fec52565aa0cf18f57d7cf5b3ac728503b8992d2d6f7d46da1d1201090902b02
?> Which user did that address log in as? :: sha256:b7bd55c11b781b0ccc43aa6e57f9dadf0660e9d1d4e27e0979ee43a407d454ae
?> Which file did that user read with sudo? (full path) :: sha256:7b5ddf499844cf05866927513cf62fee9af8c281812def040b964146b0ea87d6

## 3. The same hunt in KQL

KQL starts with a **table**, then pipes into operators: `where` filters, `summarize count() by` groups, `project` picks columns, `sort by` orders. The answer must be the same as in SPL. If it isn't, one of the two searches is wrong.

- [ ] Run the failure count in KQL and compare with SPL
- [ ] Write the success pivot in KQL
- [ ] Note one thing that is easier in each language

> Hint: In KQL the count column is called `count_`. Strings use `==` and double quotes.

```tool siem
{"q": "auth | where action == \"failure\" | summarize count() by src_ip | sort by count_ desc", "lang": "kql", "n": 2, "files": ["auth.log"], "blank": ["q"]}
```

```tool siem
{"q": "auth | where src_ip == \"203.0.113.7\" and action == \"success\" | project _time, user, auth_method", "lang": "kql", "n": 1, "files": ["auth.log"], "blank": ["q"]}
```

?> How many failed logins came from that address? (KQL count_) :: sha256:e7f6c011776e8db7cd330b54174fd76f7d0216b612387a5ffcfb81e6f0919683

## 4. SQL over the same logs

Every log is also a table. The SQL tool names each table after its file (`auth`, `access`, `security`) and makes a column for each field. SQL is the language behind most data lakes, so it is worth having in your pocket.

- [ ] Count requests per client address in the web log
- [ ] Find every request from 203.0.113.7 that got status 200
- [ ] Join the idea back to the SSH hunt: same attacker?

> Hint: `GROUP BY clientip ORDER BY hits DESC` is the SQL version of `stats count by clientip | sort - count`.

```tool sql
{"q": "SELECT clientip, COUNT(*) AS hits FROM access GROUP BY clientip ORDER BY hits DESC", "n": 3, "files": ["access.log"], "blank": ["q"]}
```

?> How many requests did 203.0.113.7 make to the web server? :: sha256:19581e27de7ced00ff1ce50b2047e7a567c76b1cbaebabe5ef03f7c3017bb5b7

## 5. Parse a raw log into fields

Searches only work on fields someone parsed. When a new log arrives that your SIEM doesn't know, you write the parser. The log parser uses `%{TYPE:name}` pieces, the same idea as Grok in log pipelines.

The pattern below parses SSH login lines. Lines it can't parse are shown in red: decide whether that is right.

- [ ] Run the pattern on auth.log
- [ ] Which line doesn't parse, and why is that fine?
- [ ] Write a second pattern for the sudo line

> Hint: The sudo line has a different shape: `sudo:   deploy : TTY=... ; COMMAND=...`. Start with `%{SYSLOGTIMESTAMP:time} %{HOSTNAME:host} sudo: +%{USER:user} : %{GREEDYDATA:rest}`.

```tool parse
{"file": "auth.log", "pat": "%{SYSLOGTIMESTAMP:time} %{HOSTNAME:host} sshd\\[%{INT:pid}\\]: %{WORD:result} %{WORD:method} for (?:invalid user )?%{USER:user} from %{IP:src_ip} port %{INT:src_port}%{GREEDYDATA:rest}", "n": 11, "blank": ["pat"]}
```

?> How many of the 12 lines does the SSH pattern parse? :: sha256:4fc82b26aecb47d2868c4efbe3581732a3e7cbcc6c2efb32062c08170a05eeb8

## 6. Hunt the web attack in SPL and KQL

The same address also hit the web server. Find the scanner, the SQL injection, and what came after it. `uri_path` is the page without the query string, which makes grouping easy.

- [ ] Which tool did the attacker use? (look at the user agent)
- [ ] Which page did they upload, and what commands did they run through it?
- [ ] Write both searches into your hunt pack

> Hint: `%20` is a space and `%27` is a quote in a URL. CyberChef's URL Decode makes the commands readable.

```tool siem
{"q": "sourcetype=access_combined useragent=sqlmap* | table _time clientip uri status", "n": 4, "files": ["access.log"], "blank": ["q"]}
```

```tool siem
{"q": "sourcetype=access_combined clientip=203.0.113.7 uri_path=/uploads/img.php | table _time uri status", "n": 3, "files": ["access.log"], "blank": ["q"]}
```

```tool siem
{"q": "access | where clientip == \"203.0.113.7\" and uri_path == \"/uploads/img.php\" | project _time, uri, status", "lang": "kql", "n": 3, "files": ["access.log"], "blank": ["q"]}
```

?> Which scanner's name is in the user agent? :: sha256:44b5a25dfad00357b11b2515f85217ecbcc302794bfa6d4fc888fa0a45cecb07
?> Which file did the attacker run commands through? (file name) :: sha256:b8e585c3a8ff8b4d84273cbebef515e96c7e40d11126a5e2f578e6a07ee20376

## 7. From hunt to rule: your first Sigma rule

A hunt you run once finds one attack. A **rule** finds the next one. Sigma is the open format for detection rules: one YAML file that converts to SPL, KQL and others.

Switch to the Windows case. On hr-pc07 Word started cmd.exe. The rule below fires on that, and on Excel or Outlook doing the same. Press **Test**, then **Convert with pySigma** to see the real converter's SPL.

- [ ] Test the rule: how many events match?
- [ ] Convert it with pySigma and compare with a search you would write by hand
- [ ] Why does PowerShell not match? (look at its parent)

> Hint: PowerShell's parent was cmd.exe, not Word. One rule per step of the chain, or a second rule for "cmd.exe starts PowerShell hidden".

```tool siem
{"q": "EventID=4688 | stats count by Computer", "n": 1, "files": ["eventlogs/Security.jsonl"], "blank": ["q"]}
```

```tool sigma
{"rule": "title: Office program starts a command shell\nid: 5a1c7e10-0b8f-4a59-9a3e-2c1f00000001\nstatus: experimental\ndescription: Word, Excel or Outlook starting cmd.exe or PowerShell is how most macro malware begins.\nlogsource:\n  product: windows\n  service: security\ndetection:\n  selection:\n    EventID: 4688\n    ParentProcessName|endswith:\n      - '\\WINWORD.EXE'\n      - '\\EXCEL.EXE'\n      - '\\OUTLOOK.EXE'\n    NewProcessName|endswith:\n      - '\\cmd.exe'\n      - '\\powershell.exe'\n  condition: selection\nfalsepositives:\n  - Office add-ins that call a script on purpose (rare; list them)\nlevel: high\ntags:\n  - attack.execution\n  - attack.t1204.002\n", "n": 1, "ls": true, "blank": ["rule"]}
```

?> How many process-creation events (4688) are in Security.jsonl? :: sha256:e629fa6598d732768f7c726b4b621285f9c3b85303900aa912017db7617d8bdb
?> How many events does the Office-starts-a-shell rule match? :: sha256:6b86b273ff34fce19d6b804eff5a3f5747ada4eaa22f1d49c01e52ddb7875b4b

## 8. Test for false positives, then tune

A rule that fires on everything is worse than no rule: people stop reading it. Test every rule twice: does it catch the attack (**true positives**) and does it stay quiet on normal work (**false positives**)?

The first rule below is deliberately noisy. Tick the events that are really malicious; the tool scores precision. Then use the tuned rules, which each describe one thing an attacker did.

- [ ] Run the noisy rule and tick the malicious events
- [ ] Run each tuned rule: each should match only attack events
- [ ] Write one tuning note per rule: what it misses, and what would make it noisy

> Hint: Good rules describe behaviour that normal users don't have: a program in C:\Users\Public, shadow copies deleted, a new member of Administrators.

```tool sigma
{"rule": "title: Any new process on a workstation\nstatus: experimental\nlogsource:\n  product: windows\n  service: security\ndetection:\n  selection:\n    EventID: 4688\n  condition: selection\nlevel: low\n", "n": 15, "ls": true, "blank": ["rule"]}
```

```tool sigma
{"rule": "title: Program started from C:\\Users\\Public\nid: 5a1c7e10-0b8f-4a59-9a3e-2c1f00000002\nstatus: experimental\ndescription: Programs should not run from the Public profile, which every user can write to.\nlogsource:\n  product: windows\n  service: security\ndetection:\n  selection:\n    EventID: 4688\n    NewProcessName|contains: '\\Users\\Public\\'\n  condition: selection\nfalsepositives:\n  - Badly packaged software installed to the Public profile (check with IT)\nlevel: high\ntags:\n  - attack.defense_evasion\n  - attack.t1036.005\n", "n": 2, "ls": true, "blank": ["rule"]}
```

```tool sigma
{"rule": "title: Shadow copies deleted\nid: 5a1c7e10-0b8f-4a59-9a3e-2c1f00000003\nstatus: experimental\ndescription: Ransomware deletes Volume Shadow Copies so files can't be restored.\nlogsource:\n  product: windows\n  service: security\ndetection:\n  selection:\n    EventID: 4688\n    CommandLine|contains|all:\n      - 'vssadmin'\n      - 'delete shadows'\n  condition: selection\nfalsepositives:\n  - Backup software clearing old copies (it rarely uses /all /quiet)\nlevel: critical\ntags:\n  - attack.impact\n  - attack.t1490\n", "n": 1, "ls": true, "blank": ["rule"]}
```

```tool sigma
{"rule": "title: Account added to the local Administrators group\nid: 5a1c7e10-0b8f-4a59-9a3e-2c1f00000004\nstatus: experimental\ndescription: A new member of Administrators on a workstation needs a change ticket.\nlogsource:\n  product: windows\n  service: security\ndetection:\n  selection:\n    EventID: 4732\n    TargetUserName: 'Administrators'\n  condition: selection\nfalsepositives:\n  - IT adding a helpdesk account during a planned change\nlevel: high\ntags:\n  - attack.persistence\n  - attack.t1136.001\n", "n": 1, "ls": true, "blank": ["rule"]}
```

?> How many events does the noisy "any new process" rule match? :: sha256:e629fa6598d732768f7c726b4b621285f9c3b85303900aa912017db7617d8bdb
?> Which command deleted the shadow copies? (program name) :: sha256:ff80268dfba624cd94529df29c7daed2f4ddd732fa3317dea937d41e10feb99a

## 9. A web rule, tested on the web log

Sigma works for web logs too. The first version below matches any `%27` (a quote) or the word UNION. `shop2_access.log` is an hour of normal shoppers: run the noisy version on it and see who it accuses. The tuned version keys on the tool's name and on `UNION%20SELECT`.

- [ ] Test both versions on both web logs
- [ ] Which requests does the noisy version catch that the tuned one doesn't? Are they attacks?
- [ ] Write the tuning note: when would the tuned rule miss an attack?

> Hint: An attacker can change the user agent in one flag. Keep a second rule for the injection itself, not just the tool.

```tool sigma
{"rule": "title: SQL injection in a web request\nstatus: experimental\nlogsource:\n  category: webserver\ndetection:\n  keywords:\n    - '%27'\n    - 'UNION'\n    - 'sqlmap'\n  condition: keywords\nlevel: high\n", "n": 6, "ls": true, "blank": ["rule"]}
```

```tool sigma
{"rule": "title: SQL injection tool or UNION SELECT in a web request\nid: 5a1c7e10-0b8f-4a59-9a3e-2c1f00000005\nstatus: experimental\nlogsource:\n  category: webserver\ndetection:\n  tool:\n    - 'sqlmap/'\n  union:\n    - 'UNION%20SELECT'\n  condition: tool or union\nfalsepositives:\n  - Your own authorised scans (exclude the scanner's address in the tuning notes)\nlevel: high\ntags:\n  - attack.initial_access\n  - attack.t1190\n", "n": 4, "ls": true, "blank": ["rule"]}
```

?> How many web requests does the noisy version match? :: sha256:e7f6c011776e8db7cd330b54174fd76f7d0216b612387a5ffcfb81e6f0919683
?> How many web requests does the tuned rule match? :: sha256:4b227777d4dd1fc61c6f884f48641d02b4d121d3fd328cb08b5531fcacdabf8a

## 10. Hand in your hunt pack and rule pack

Your hand-in is this notebook, with every search and rule saved in it. Before you hand in:

| Piece | What it must have |
|---|---|
| Hunt pack | The SSH hunt and the web hunt, each in SPL and KQL, with the answer they give |
| SQL | One grouping query and one sentence on what it shows |
| Rule pack | Five Sigma rules (four Windows, one web), each tested, with a tuning note |
| Conversion | Each rule converted with pySigma, and one difference you noticed |

- [ ] Every question answered
- [ ] Every rule shows its test result
- [ ] A tuning note under each rule
- [ ] Press Hand in

**Rubric**

| Criterion | Points |
|---|---|
| Hunt pack: the brute-force hunt in SPL and in KQL, both giving the same answer | 4 |
| Hunt pack: the web attack hunt in SPL and KQL, with the attacker's IP and what they reached | 4 |
| SQL: one query that groups and counts, with the result explained | 2 |
| Sigma: four Windows rules that each match the attack and nothing else | 6 |
| Sigma: the web rule, with its false-positive test and tuning note | 3 |
| Conversion: each rule converted with pySigma, with one difference from your hand-written search noted | 3 |
| Tuning notes: what each rule misses, and what would make it noisy | 3 |